In [ ]:
# Step 1: Load the final human-reviewed annotation dataset

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd().parent
OUTPUT_DIR = PROJECT_DIR / "outputs"

ANNOTATION_FILE = (
    OUTPUT_DIR / "step16_formal_annotation_sample_human_reviewed.xlsx"
)

ann = pd.read_excel(ANNOTATION_FILE)

print("Annotation dataset shape:", ann.shape)

print("\nColumns:")
print(ann.columns.tolist())

display(ann.head())

In [ ]:
# Step 2: Validate annotation integrity and label values

VALID_MENTION_LABELS = {
    "yes",
    "no"
}

INSTITUTIONAL_ROLE_LABELS = {
    "protective_supportive",
    "partial_limited_support",
    "bureaucratic_inaccessible",
    "dismissive_disbelieving",
    "harmful_retraumatising",
    "neutral_descriptive",
    "absent_unavailable",
    "unclear"
}

JUSTICE_DIMENSION_LABELS = {
    "safety_prevention",
    "recognition",
    "voice",
    "dignity",
    "consequences",
    "connectedness_recovery",
    "unclear_na"
}

EVALUATION_LABELS = {
    "positive",
    "negative",
    "mixed",
    "neutral_descriptive",
    "unclear"
}

annotation_cols = [
    "valid_mention",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction"
]

print("=== BASIC INTEGRITY ===")
print("Rows:", len(ann))
print("Unique annotation IDs:", ann["annotation_id"].nunique())
print("Duplicate annotation IDs:", ann["annotation_id"].duplicated().sum())

print("\nMissing values in annotation fields:")
print(ann[annotation_cols].isna().sum())

print("\n=== OBSERVED LABELS ===")
for col in annotation_cols:
    print(f"\n{col}:")
    print(sorted(ann[col].dropna().astype(str).unique()))

expected_labels = {
    "valid_mention": VALID_MENTION_LABELS,
    "institutional_role": INSTITUTIONAL_ROLE_LABELS,
    "justice_dimension": JUSTICE_DIMENSION_LABELS,
    "evaluative_direction": EVALUATION_LABELS
}

print("\n=== ILLEGAL LABEL CHECK ===")

all_labels_valid = True

for col, allowed in expected_labels.items():
    observed = set(ann[col].dropna().astype(str))
    illegal = observed - allowed

    print(f"{col}: {sorted(illegal) if illegal else 'None'}")

    if illegal:
        all_labels_valid = False

print("\nAll annotation labels valid:", all_labels_valid)

### Annotation data validation

The final human-reviewed dataset contains 90 unique annotation records with no duplicate annotation IDs and no missing values in the four annotation fields. All observed labels conform to the predefined annotation scheme.

The dataset is therefore treated as the final analysis-ready annotation dataset. No further automatic cleaning, relabelling, or modification is applied before substantive analysis.

In [ ]:
# Step 3: Examine valid institutional mentions

# Overall counts and proportions
valid_overall = (
    ann["valid_mention"]
    .value_counts()
    .rename_axis("valid_mention")
    .to_frame("count")
)

valid_overall["proportion"] = (
    valid_overall["count"] / len(ann)
).round(3)

print("=== OVERALL VALID MENTIONS ===")
display(valid_overall)


# Counts by institutional category
valid_by_category = pd.crosstab(
    ann["category"],
    ann["valid_mention"]
)

print("\n=== VALID MENTIONS BY CATEGORY ===")
display(valid_by_category)


# Proportions within each institutional category
valid_by_category_prop = pd.crosstab(
    ann["category"],
    ann["valid_mention"],
    normalize="index"
).round(3)

print("\n=== VALID-MENTION PROPORTIONS BY CATEGORY ===")
display(valid_by_category_prop)

### Valid-mention results and analysis decision

Manual review identified 73 valid institutional mentions out of the 90 sampled KWIC records (81.1%).

Validity was similar for police and legal records, with 25 of 30 cases retained in each category (83.3%). Support-sector records had a slightly lower validation rate, with 23 of 30 cases retained (76.7%).

The invalid cases mainly represent retrieval hits that did not constitute substantive institutional descriptions under the annotation rules. Because the sample was constructed purposively rather than as a probability sample, these validation rates are treated as sample diagnostics rather than estimates of corpus-wide precision.

Subsequent analyses of institutional role, justice dimension, and evaluative direction are restricted to records coded as `valid_mention = yes`. Invalid records are retained in the original dataset for traceability but excluded from substantive annotation comparisons.

In [ ]:
# Step 4: Create the valid substantive annotation subset

ann_valid = ann[
    ann["valid_mention"] == "yes"
].copy()

print("Valid annotation records:", len(ann_valid))

print("\nValid records by category:")
print(
    ann_valid["category"]
    .value_counts()
)

print("\nUnique transcripts represented:")
print(
    ann_valid["transcript_id"]
    .nunique()
)

In [ ]:
# Step 5: Examine institutional role distributions

# Overall institutional-role distribution
role_overall = (
    ann_valid["institutional_role"]
    .value_counts()
    .rename_axis("institutional_role")
    .to_frame("count")
)

role_overall["proportion"] = (
    role_overall["count"] / len(ann_valid)
).round(3)

print("=== OVERALL INSTITUTIONAL ROLE DISTRIBUTION ===")
display(role_overall)


# Counts by institutional category
role_by_category = pd.crosstab(
    ann_valid["category"],
    ann_valid["institutional_role"]
)

print("\n=== INSTITUTIONAL ROLE BY CATEGORY: COUNTS ===")
display(role_by_category)


# Within-category proportions
role_by_category_prop = pd.crosstab(
    ann_valid["category"],
    ann_valid["institutional_role"],
    normalize="index"
).round(3)

print("\n=== INSTITUTIONAL ROLE BY CATEGORY: PROPORTIONS ===")
display(role_by_category_prop)

### Institutional-role patterns

Across the 73 valid annotations, `protective_supportive` was the most frequent institutional role (21 cases; 28.8%), followed by `partial_limited_support` (19.2%) and `neutral_descriptive` (16.4%). Negative or constrained institutional roles were nevertheless distributed across several distinct forms, including bureaucratic inaccessibility, institutional absence, harmful or retraumatising responses, and dismissal or disbelief.

The three institutional categories showed different role profiles.

- **Support-sector** narratives were most strongly characterised by `protective_supportive` responses (11/23; 47.8%). No support-sector cases in the annotated sample were coded as `harmful_retraumatising` or `dismissive_disbelieving`, although limited support, service absence, and access barriers remained present.
- **Police** narratives were more mixed. Protective responses accounted for 7/25 cases (28.0%), while `partial_limited_support` (24.0%), `absent_unavailable` (16.0%), and `harmful_retraumatising` (16.0%) were also prominent.
- **Legal** narratives contained comparatively more `neutral_descriptive` cases (9/25; 36.0%). Protective responses were less frequent (3/25; 12.0%), while bureaucratic inaccessibility and limited support each accounted for 16.0%.

These patterns suggest that institutional differences are not captured adequately by a simple positive/negative distinction. The annotated passages described institutions through multiple forms of support, limitation, absence, procedural constraint, and harm. Because the annotation sample was purposively constructed, these proportions are interpreted as patterns within the annotated material rather than population estimates.

In [ ]:
# Step 6: Examine justice-dimension distributions

# Overall justice-dimension distribution
justice_overall = (
    ann_valid["justice_dimension"]
    .value_counts()
    .rename_axis("justice_dimension")
    .to_frame("count")
)

justice_overall["proportion"] = (
    justice_overall["count"] / len(ann_valid)
).round(3)

print("=== OVERALL JUSTICE-DIMENSION DISTRIBUTION ===")
display(justice_overall)


# Counts by institutional category
justice_by_category = pd.crosstab(
    ann_valid["category"],
    ann_valid["justice_dimension"]
)

print("\n=== JUSTICE DIMENSION BY CATEGORY: COUNTS ===")
display(justice_by_category)


# Within-category proportions
justice_by_category_prop = pd.crosstab(
    ann_valid["category"],
    ann_valid["justice_dimension"],
    normalize="index"
).round(3)

print("\n=== JUSTICE DIMENSION BY CATEGORY: PROPORTIONS ===")
display(justice_by_category_prop)

### Justice-dimension patterns

Across the 73 valid annotations, `safety_prevention` was the most frequent justice dimension (26 cases; 35.6%), followed by `consequences` (20.5%), `connectedness_recovery` (15.1%), `voice` (13.7%), and `recognition` (11.0%). `Dignity` appeared relatively infrequently in the annotated sample (4.1%).

The distribution differed substantially across institutional categories. Police narratives were primarily associated with `safety_prevention` (11/25; 44.0%), with `consequences` forming the second largest dimension (20.0%). Legal narratives were most strongly associated with `consequences` (9/25; 36.0%) and also contained comparatively more `voice` cases (20.0%). Support-sector narratives were concentrated around `safety_prevention` (11/23; 47.8%) and `connectedness_recovery` (7/23; 30.4%).

This differentiation suggests that the institutional categories show different profiles across the primary justice dimensions assigned to the annotated passages. Police and support services were both strongly connected to safety, but support-sector narratives additionally emphasised recovery and connectedness, whereas legal narratives were more closely associated with consequences and opportunities for voice. These patterns are descriptive of the purposively selected annotation sample rather than estimates of prevalence in the full corpus.

In [ ]:
# Step 7: Examine evaluative-direction distributions

# Overall evaluative-direction distribution
evaluation_overall = (
    ann_valid["evaluative_direction"]
    .value_counts()
    .rename_axis("evaluative_direction")
    .to_frame("count")
)

evaluation_overall["proportion"] = (
    evaluation_overall["count"] / len(ann_valid)
).round(3)

print("=== OVERALL EVALUATIVE-DIRECTION DISTRIBUTION ===")
display(evaluation_overall)


# Counts by institutional category
evaluation_by_category = pd.crosstab(
    ann_valid["category"],
    ann_valid["evaluative_direction"]
)

print("\n=== EVALUATIVE DIRECTION BY CATEGORY: COUNTS ===")
display(evaluation_by_category)


# Within-category proportions
evaluation_by_category_prop = pd.crosstab(
    ann_valid["category"],
    ann_valid["evaluative_direction"],
    normalize="index"
).round(3)

print("\n=== EVALUATIVE DIRECTION BY CATEGORY: PROPORTIONS ===")
display(evaluation_by_category_prop)

### Evaluative-direction patterns

Across the 73 valid annotations, negative evaluations were the most frequent (26 cases; 35.6%), followed by positive evaluations (30.1%), neutral descriptions (21.9%), and mixed evaluations (12.3%). The overall distribution therefore does not indicate a uniformly negative or positive representation of institutions.

Clear differences emerged across institutional categories. Police narratives were predominantly negative (12/25; 48.0%), although positive experiences were also present (28.0%). Legal narratives similarly contained a high proportion of negative evaluations (11/25; 44.0%), but were distinguished by a substantial proportion of neutral or descriptive accounts (36.0%) and relatively few positive evaluations (12.0%). In contrast, support-sector narratives were predominantly positive (12/23; 52.2%), with negative evaluations accounting for only 13.0%.

These evaluative patterns reinforce the institutional-role analysis while providing a broader summary of direction. However, evaluative direction is treated as a complementary rather than standalone measure, since similar positive or negative evaluations may reflect different institutional roles and different dimensions of justice.

In [ ]:
# Step 8: Examine institutional role × evaluative direction

role_eval_counts = pd.crosstab(
    ann_valid["institutional_role"],
    ann_valid["evaluative_direction"]
)

print("=== INSTITUTIONAL ROLE × EVALUATIVE DIRECTION: COUNTS ===")
display(role_eval_counts)


role_eval_prop = pd.crosstab(
    ann_valid["institutional_role"],
    ann_valid["evaluative_direction"],
    normalize="index"
).round(3)

print("\n=== INSTITUTIONAL ROLE × EVALUATIVE DIRECTION: ROW PROPORTIONS ===")
display(role_eval_prop)

### Relationship between institutional role and evaluative direction

The cross-tabulation shows a strong relationship between institutional-role coding and evaluative direction. All `protective_supportive` cases were evaluated positively, while all `bureaucratic_inaccessible` and `dismissive_disbelieving` cases were negative. `Neutral_descriptive` institutional roles corresponded entirely to neutral evaluations.

Greater variation appeared in less categorical institutional roles. `partial_limited_support` was divided between mixed and negative evaluations, while `absent_unavailable` included neutral, negative, and positive cases.

This relationship is interpreted primarily as a coherence check rather than an independent substantive finding, because institutional role and evaluative direction are conceptually related within the annotation framework. Evaluative direction provides a broad summary of orientation, whereas institutional role retains more specific information about how an institution was described as supporting, limiting, failing, or harming survivors.

In [ ]:
# Step 9: Examine justice dimension × institutional role

justice_role_counts = pd.crosstab(
    ann_valid["justice_dimension"],
    ann_valid["institutional_role"]
)

print("=== JUSTICE DIMENSION × INSTITUTIONAL ROLE: COUNTS ===")
display(justice_role_counts)

In [ ]:
# Step 10: Summarise dominant institutional roles within each justice dimension

justice_role_long = (
    ann_valid
    .groupby(
        ["justice_dimension", "institutional_role"]
    )
    .size()
    .reset_index(name="count")
)

justice_totals = (
    justice_role_long
    .groupby("justice_dimension")["count"]
    .transform("sum")
)

justice_role_long["proportion_within_dimension"] = (
    justice_role_long["count"] / justice_totals
).round(3)

justice_role_summary = (
    justice_role_long
    .sort_values(
        ["justice_dimension", "count"],
        ascending=[True, False]
    )
)

print("=== INSTITUTIONAL ROLES WITHIN EACH JUSTICE DIMENSION ===")
display(justice_role_summary)

### Justice dimensions and institutional roles

Cross-tabulation of justice dimensions and institutional roles shows that the same dimension of justice could be associated with substantially different institutional responses.

`Safety_prevention`, the most frequent justice dimension, was most commonly associated with a `protective_supportive` role (15/26; 57.7%). However, the remaining safety-related cases included institutional absence, bureaucratic barriers, limited support, dismissal, and neutral responses. Safety therefore appeared both as something institutions could actively provide and as a need that could remain only partially met or obstructed.

Other justice dimensions showed different patterns. `Consequences` was most often associated with `partial_limited_support` (6/15; 40.0%) or `neutral_descriptive` institutional roles (5/15; 33.3%), suggesting that accounts of accountability frequently concerned incomplete or procedural institutional responses. `Recognition` showed a particularly contrasting pattern: half of the cases were `protective_supportive` (4/8), while others involved dismissal, disbelief, or harm. `Voice` was primarily associated with neutral/descriptive, harmful, or limited institutional roles rather than clearly protective responses.

The small number of `dignity` cases (n=3) prevents meaningful interpretation of its internal distribution. More generally, low-frequency combinations are treated cautiously.

These results indicate that justice dimensions describe what was at stake in survivors' accounts, whereas institutional-role coding captures how institutions responded to those needs. The two dimensions therefore provide complementary rather than interchangeable information.

In [ ]:
# Step 11: Examine NMF topic coverage in valid annotations

topic_counts = (
    ann_valid["dominant_topic"]
    .value_counts()
    .sort_index()
    .rename_axis("dominant_topic")
    .to_frame("count")
)

topic_counts["proportion"] = (
    topic_counts["count"] / len(ann_valid)
).round(3)

print("=== NMF TOPIC COVERAGE IN VALID ANNOTATIONS ===")
display(topic_counts)


topic_by_category = pd.crosstab(
    ann_valid["dominant_topic"],
    ann_valid["category"]
)

print("\n=== NMF TOPIC × INSTITUTIONAL CATEGORY ===")
display(topic_by_category)

### NMF topic coverage after manual validation

The 73 valid annotations retained substantial coverage of the major NMF topics used during sample construction. Topics 1, 5 and 6 each contained 16–17 valid cases, Topic 4 contained 11 cases, and Topic 8 contained 9 cases. Topic 7 was represented by only four valid cases, while Topic 2 was no longer represented after manual validation.

The institutional composition of these topics broadly reflects the NMF-informed sampling design: Topic 1 was concentrated in police records, Topic 4 in support-sector records, and Topic 6 in legal records, while Topic 5 contained cases from all three institutional categories.

These distributions are therefore treated as a coverage check rather than an independent finding about topic prevalence or institutional association. Subsequent integration uses NMF topics as computational patterns to be compared with manual interpretation, not as substantive labels or statistically independent categories.

Because Topic 7 has limited coverage and Topic 2 has no valid annotated cases, detailed interpretation of these topics is avoided in the annotation-level integration.

In [ ]:
# Step 12: Summarise manual annotation patterns within major NMF topics

topic_n = (
    ann_valid["dominant_topic"]
    .value_counts()
)

major_topics = topic_n[
    topic_n >= 8
].index.tolist()

ann_major_topics = ann_valid[
    ann_valid["dominant_topic"].isin(major_topics)
].copy()

print("Major topics retained:", sorted(major_topics))

print("\nValid cases represented:")
print(
    ann_major_topics["dominant_topic"]
    .value_counts()
    .sort_index()
)


# Most common justice dimension within each topic
topic_justice = (
    ann_major_topics
    .groupby(["dominant_topic", "justice_dimension"])
    .size()
    .reset_index(name="count")
)

topic_justice["topic_total"] = (
    topic_justice
    .groupby("dominant_topic")["count"]
    .transform("sum")
)

topic_justice["proportion"] = (
    topic_justice["count"]
    / topic_justice["topic_total"]
).round(3)

print("\n=== JUSTICE DIMENSIONS WITHIN MAJOR NMF TOPICS ===")
display(
    topic_justice.sort_values(
        ["dominant_topic", "count"],
        ascending=[True, False]
    )
)


# Most common institutional role within each topic
topic_role = (
    ann_major_topics
    .groupby(["dominant_topic", "institutional_role"])
    .size()
    .reset_index(name="count")
)

topic_role["topic_total"] = (
    topic_role
    .groupby("dominant_topic")["count"]
    .transform("sum")
)

topic_role["proportion"] = (
    topic_role["count"]
    / topic_role["topic_total"]
).round(3)

print("\n=== INSTITUTIONAL ROLES WITHIN MAJOR NMF TOPICS ===")
display(
    topic_role.sort_values(
        ["dominant_topic", "count"],
        ascending=[True, False]
    )
)

### Interpretation of NMF–annotation alignment

The comparison shows partial rather than one-to-one alignment between the computational topics and the manually interpreted categories.

Some topics displayed relatively clear substantive concentration. In Topic 1, half of the valid cases were coded as safety/prevention, followed by consequences (31.2%). Topic 4 was particularly concentrated: all 11 valid cases concerned either safety/prevention (63.6%) or connectedness/recovery (36.4%). These topics therefore appear to capture relatively coherent institutional contexts that correspond to particular justice concerns.

Other topics remained considerably more heterogeneous. Topic 5 contained all six justice dimensions and a wide range of institutional roles, while Topics 6 and 8 also showed dispersed manual interpretations. This indicates that the lexical patterns identified by NMF do not map directly onto theoretically defined justice dimensions or institutional roles.

The NMF topics are therefore retained as exploratory computational structures rather than treated as substantive categories in their own right. Manual annotation provides the interpretive layer required to establish how these textual patterns relate to institutional experiences of justice and injustice.

In [ ]:
# Step 13: Build institution-level synthesis profiles

categories = ["police", "legal", "support_sector"]

synthesis_rows = []

for category in categories:
    subset = ann_valid[ann_valid["category"] == category]
    n = len(subset)

    # Institutional role
    role_counts = subset["institutional_role"].value_counts()
    top_roles = "; ".join(
        [
            f"{label} ({count}/{n}, {count/n:.1%})"
            for label, count in role_counts.head(3).items()
        ]
    )

    # Justice dimension
    justice_counts = subset["justice_dimension"].value_counts()
    top_justice = "; ".join(
        [
            f"{label} ({count}/{n}, {count/n:.1%})"
            for label, count in justice_counts.head(3).items()
        ]
    )

    # Evaluative direction
    eval_counts = subset["evaluative_direction"].value_counts()
    evaluation_profile = "; ".join(
        [
            f"{label} ({count}/{n}, {count/n:.1%})"
            for label, count in eval_counts.items()
        ]
    )

    synthesis_rows.append({
        "category": category,
        "valid_n": n,
        "top_institutional_roles": top_roles,
        "top_justice_dimensions": top_justice,
        "evaluation_profile": evaluation_profile
    })

institution_profiles = pd.DataFrame(synthesis_rows)

print("=== INSTITUTION-LEVEL SYNTHESIS ===")
display(institution_profiles)

### Interpretation of institution-level profiles

The synthesis indicates distinct descriptive profiles across the three institutional categories. Police-related narratives were most strongly associated with safety and prevention (44.0%), but evaluations were predominantly negative (48.0%), alongside a substantial positive component (28.0%). This suggests that police encounters were closely connected to survivors' safety needs, while the perceived quality and outcome of institutional responses varied considerably.

Legal narratives were more strongly oriented towards consequences (36.0%) and voice (20.0%). Neutral/descriptive institutional roles were particularly common (36.0%), while negative evaluations remained prominent (44.0%). This profile suggests that legal institutions were frequently described through procedural processes and their capacity to produce consequences, alongside experiences of institutional limitation or inaccessibility.

Support-sector narratives showed the clearest supportive profile. Protective/supportive roles accounted for 47.8% of valid cases, positive evaluations for 52.2%, and the dominant justice dimensions were safety/prevention (47.8%) and connectedness/recovery (30.4%). Support organisations therefore appeared more consistently in narratives concerning both immediate safety and longer-term recovery, although limited or inaccessible support was also present.

These profiles are descriptive rather than population-level estimates. They summarise the manually annotated sample and are used to identify patterns for subsequent targeted close reading.

In [ ]:
# Step 14: Identify recurrent institution–justice–role patterns

pattern_summary = (
    ann_valid
    .groupby(
        ["category", "justice_dimension", "institutional_role"],
        observed=True
    )
    .size()
    .reset_index(name="count")
)

# Calculate proportion within each institutional category
category_totals = (
    ann_valid
    .groupby("category", observed=True)
    .size()
    .rename("category_total")
    .reset_index()
)

pattern_summary = pattern_summary.merge(
    category_totals,
    on="category",
    how="left"
)

pattern_summary["proportion_within_category"] = (
    pattern_summary["count"] / pattern_summary["category_total"]
)

# Retain recurrent patterns rather than isolated single cases
recurrent_patterns = (
    pattern_summary[
        pattern_summary["count"] >= 2
    ]
    .sort_values(
        ["category", "count"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("=== RECURRENT INSTITUTION × JUSTICE × ROLE PATTERNS ===")
display(recurrent_patterns)

In [ ]:
## Exploratory social-position feasibility check
# Step 15: Load and inspect participant metadata

from pathlib import Path
import pandas as pd

DATA_DIR = PROJECT_DIR
DATALIST_FILE = DATA_DIR / "dataset" / "DataList.xlsx"

# The first Excel row contains grouped section headings.
# The second row contains the actual variable names.
metadata = pd.read_excel(
    DATALIST_FILE,
    header=1
)

print("Metadata shape:", metadata.shape)

print("\nColumns:")
print(metadata.columns.tolist())

print("\nFirst rows:")
display(metadata.head())

print("\nMissing values by column:")
missing_summary = (
    metadata.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_n")
)

missing_summary["missing_proportion"] = (
    missing_summary["missing_n"] / len(metadata)
).round(3)

display(missing_summary)

In [ ]:
# Step 16: Link valid annotation transcripts to participant metadata

# Unique transcripts represented in valid annotations
valid_transcripts = (
    ann_valid[["transcript_id"]]
    .drop_duplicates()
    .copy()
)

print("Unique valid transcripts:", len(valid_transcripts))

# Merge transcript IDs with participant metadata
metadata_linked = valid_transcripts.merge(
    metadata,
    left_on="transcript_id",
    right_on="Initial Participant ID",
    how="left",
    indicator=True
)

print("\nMerge status:")
print(metadata_linked["_merge"].value_counts())

print("\nMatched transcripts:")
print(
    (metadata_linked["_merge"] == "both").sum()
)

print("\nUnmatched transcripts:")
unmatched = metadata_linked.loc[
    metadata_linked["_merge"] != "both",
    "transcript_id"
].tolist()

print(unmatched)

print("\nLinked metadata shape:")
print(metadata_linked.shape)

In [ ]:
# Step 16: Link valid annotation transcripts to participant metadata

# Unique transcripts represented in valid annotations
valid_transcripts = (
    ann_valid[["transcript_id"]]
    .drop_duplicates()
    .copy()
)

print("Unique valid transcripts:", len(valid_transcripts))

# Merge transcript IDs with participant metadata
metadata_linked = valid_transcripts.merge(
    metadata,
    left_on="transcript_id",
    right_on="Initial Participant ID",
    how="left",
    indicator=True
)

print("\nMerge status:")
print(metadata_linked["_merge"].value_counts())

print("\nMatched transcripts:")
print(
    (metadata_linked["_merge"] == "both").sum()
)

print("\nUnmatched transcripts:")
unmatched = metadata_linked.loc[
    metadata_linked["_merge"] != "both",
    "transcript_id"
].tolist()

print(unmatched)

print("\nLinked metadata shape:")
print(metadata_linked.shape)

## 17. Exploratory social-position feasibility audit

The primary analysis above addresses the main research question by examining how survivors describe police, legal, and support-sector institutions through institutional roles, justice dimensions, evaluative direction, and their relationships with the NMF-derived patterns.

The original project also proposed a secondary, exploratory question:

> Do these narratives vary by social position, such as ethnicity, disability, age, nationality, immigration status, or family circumstances?

Before conducting any group comparison, the manually analysed cases are linked to the original DataList metadata. This section assesses whether selected social-position variables have sufficient coverage and meaningful group sizes to support limited exploratory comparison.

Only transcripts represented in the valid manual annotations are considered. The purpose of this stage is therefore a **feasibility audit rather than hypothesis testing**. Variables with substantial missingness, highly fragmented categories, or very small groups will not be used for comparative claims.

In [ ]:
# Step 17: Feasibility audit for social-position variables
# Only participants represented in valid annotations

social_vars = [
    "Age",
    "Identified Ethnicity",
    "BME?",
    "Nationality",
    "NATIONALITY SUMMARY",
    "ANY DECLARED PHYSICAL DISABILITY",
    "ANY DECLARED MENTAL HEALTH  OR LEARNING NEED (nb. Not distinguishing disability)",
    "Children"
]

for col in social_vars:
    print("\n" + "=" * 70)
    print(col)
    print("=" * 70)

    print("\nMissing:")
    print(
        f"{metadata_linked[col].isna().sum()} / {len(metadata_linked)} "
        f"({metadata_linked[col].isna().mean():.1%})"
    )

    print("\nDistribution:")
    print(
        metadata_linked[col]
        .fillna("MISSING")
        .astype(str)
        .value_counts(dropna=False)
    )

### Social-position feasibility decision

The feasibility audit showed substantial variation in the analytical usefulness of the available social-position variables.

Age was not retained for comparative analysis because the metadata combine exact ages, broad age ranges, and categorical descriptions, making consistent recoding difficult without introducing additional assumptions. Ethnicity, BME status, nationality, and physical disability were well recorded, but the analytically relevant minority groups were small within the 60 transcripts represented in the valid annotation sample. These variables are therefore retained primarily as contextual information for targeted close reading rather than for systematic subgroup comparison.

Two variables provide more viable group sizes for limited exploratory comparison. The mental-health/learning-need indicator divides the sample into 40 participants coded as having a declared need and 20 without, while parental status can be represented through the `Children` variable after harmonising the `No` and `no` values, producing groups of 45 and 15.

The secondary research question is therefore explored primarily through mental-health/learning-need status and parental status. Any observed differences are treated as descriptive signals rather than population-level group effects. Ethnicity/BME status, nationality, and physical disability may be revisited during close reading where they provide relevant contextual explanation.

In [ ]:
# Step 18: Attach selected social-position variables to valid annotations

selected_metadata = metadata_linked[
    [
        "Initial Participant ID",
        "ANY DECLARED MENTAL HEALTH  OR LEARNING NEED (nb. Not distinguishing disability)",
        "Children",
        "BME?",
        "NATIONALITY SUMMARY",
        "ANY DECLARED PHYSICAL DISABILITY"
    ]
].copy()

# Harmonise parental-status coding
selected_metadata["Children"] = (
    selected_metadata["Children"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "yes": "Yes",
        "no": "No"
    })
)

# Rename variables for analysis
selected_metadata = selected_metadata.rename(
    columns={
        "Initial Participant ID": "transcript_id",
        "ANY DECLARED MENTAL HEALTH  OR LEARNING NEED (nb. Not distinguishing disability)": "mental_health_learning_need",
        "Children": "parental_status",
        "BME?": "bme_status",
        "NATIONALITY SUMMARY": "nationality_summary",
        "ANY DECLARED PHYSICAL DISABILITY": "physical_disability"
    }
)

# Merge onto valid annotation records
ann_social = ann_valid.merge(
    selected_metadata,
    on="transcript_id",
    how="left",
    validate="many_to_one"
)

print("Annotation records after metadata merge:", len(ann_social))

print("\nUnique transcripts:")
print(ann_social["transcript_id"].nunique())

print("\nMissing values in selected social-position variables:")
print(
    ann_social[
        [
            "mental_health_learning_need",
            "parental_status",
            "bme_status",
            "nationality_summary",
            "physical_disability"
        ]
    ].isna().sum()
)

print("\nMental-health / learning-need distribution across annotations:")
print(
    ann_social["mental_health_learning_need"]
    .value_counts(dropna=False)
)

print("\nParental-status distribution across annotations:")
print(
    ann_social["parental_status"]
    .value_counts(dropna=False)
)

### Analytical unit for exploratory subgroup comparison

The linked dataset contains 73 valid annotation records from 60 unique transcripts. Some participants contribute valid annotations to more than one institutional category, so annotation-level records are not statistically independent observations of different survivors.

The exploratory social-position analysis therefore distinguishes between two levels:

- **participant level**, used to describe subgroup composition and institutional coverage;
- **annotation level**, used only to explore how institutional roles, justice dimensions, and evaluations are distributed within the annotated narratives.

Annotation-level proportions are interpreted as patterns in the selected narrative records rather than as estimates of participant-level prevalence or group effects. No inferential statistical tests are applied.

In [ ]:
# Step 19: Check participant-level and institutional coverage by social-position group

# One row per participant
participant_social = (
    ann_social[
        [
            "transcript_id",
            "mental_health_learning_need",
            "parental_status"
        ]
    ]
    .drop_duplicates()
)

print("=== PARTICIPANT-LEVEL GROUP SIZES ===")

print("\nMental-health / learning-need status:")
print(
    participant_social["mental_health_learning_need"]
    .value_counts()
)

print("\nParental status:")
print(
    participant_social["parental_status"]
    .value_counts()
)


# Participant-level institutional coverage
mh_coverage = (
    ann_social[
        [
            "transcript_id",
            "mental_health_learning_need",
            "category"
        ]
    ]
    .drop_duplicates()
)

print("\n=== MENTAL-HEALTH / LEARNING-NEED × INSTITUTIONAL COVERAGE ===")
display(
    pd.crosstab(
        mh_coverage["mental_health_learning_need"],
        mh_coverage["category"]
    )
)


parent_coverage = (
    ann_social[
        [
            "transcript_id",
            "parental_status",
            "category"
        ]
    ]
    .drop_duplicates()
)

print("\n=== PARENTAL STATUS × INSTITUTIONAL COVERAGE ===")
display(
    pd.crosstab(
        parent_coverage["parental_status"],
        parent_coverage["category"]
    )
)

### Institutional coverage and comparison strategy

The two retained social-position variables show sufficient coverage to support limited exploratory analysis, but their institutional composition is not identical across groups.

Participants with a declared mental-health or learning need contributed proportionally more valid support-sector and police records, while those without such a declared need were relatively more concentrated in legal narratives. Parental-status groups also differed in institutional coverage, with the smaller non-parent group represented by relatively few cases in each institutional category.

Because institutional category is itself strongly associated with institutional role, justice dimension, and evaluative direction, pooling all annotations across institutions could confound social-position differences with differences in institutional composition. Subsequent exploratory comparisons are therefore stratified by institutional category.

Given the small subgroup sizes, particularly within some institution-by-group combinations, the analysis remains descriptive. Counts are reported alongside within-group proportions, and no inferential tests are used.

In [ ]:
# Step 20: Compare evaluative direction by social position within institutional categories

def stratified_evaluation_summary(data, group_var):

    counts = pd.crosstab(
        [data["category"], data[group_var]],
        data["evaluative_direction"]
    )

    proportions = pd.crosstab(
        [data["category"], data[group_var]],
        data["evaluative_direction"],
        normalize="index"
    ).round(3)

    print(f"=== {group_var.upper()} × EVALUATION: COUNTS ===")
    display(counts)

    print(f"\n=== {group_var.upper()} × EVALUATION: WITHIN-GROUP PROPORTIONS ===")
    display(proportions)


print("### MENTAL-HEALTH / LEARNING-NEED STATUS ###\n")
stratified_evaluation_summary(
    ann_social,
    "mental_health_learning_need"
)

print("\n\n### PARENTAL STATUS ###\n")
stratified_evaluation_summary(
    ann_social,
    "parental_status"
)

### Exploratory evaluation patterns by social position

The stratified evaluation comparison did not reveal a single consistent social-position pattern across all three institutional categories.

Differences by mental-health/learning-need status were relatively modest. Legal evaluations were similar across the two groups, while police narratives among participants with a declared mental-health or learning need contained somewhat fewer negative and more positive records. Support-sector narratives remained predominantly positive in both groups, although the group without a declared need was represented by only six records.

Parental status showed somewhat larger descriptive differences, particularly in police narratives, where negative evaluations were more frequent among the smaller non-parent group. Similar directional differences appeared in legal and support-sector records. However, institution-specific non-parent subsamples were small (5–7 records), making these patterns highly sensitive to individual cases.

These results are therefore treated as exploratory signals for subsequent qualitative inspection rather than evidence of systematic group differences.

In [ ]:
# Step 21: Summarise subgroup profiles within institutional categories

def build_subgroup_profiles(data, group_var):

    rows = []

    for (category, group), subset in data.groupby(
        ["category", group_var],
        observed=True
    ):
        n_records = len(subset)
        n_transcripts = subset["transcript_id"].nunique()

        role_counts = subset["institutional_role"].value_counts()
        justice_counts = subset["justice_dimension"].value_counts()

        top_role = role_counts.index[0]
        top_role_n = role_counts.iloc[0]

        top_justice = justice_counts.index[0]
        top_justice_n = justice_counts.iloc[0]

        rows.append({
            "category": category,
            group_var: group,
            "annotation_n": n_records,
            "transcript_n": n_transcripts,
            "top_institutional_role": top_role,
            "role_n": top_role_n,
            "role_proportion": round(top_role_n / n_records, 3),
            "top_justice_dimension": top_justice,
            "justice_n": top_justice_n,
            "justice_proportion": round(top_justice_n / n_records, 3)
        })

    return pd.DataFrame(rows)


mh_profiles = build_subgroup_profiles(
    ann_social,
    "mental_health_learning_need"
)

parent_profiles = build_subgroup_profiles(
    ann_social,
    "parental_status"
)

print("=== MENTAL-HEALTH / LEARNING-NEED SUBGROUP PROFILES ===")
display(mh_profiles)

print("\n=== PARENTAL-STATUS SUBGROUP PROFILES ===")
display(parent_profiles)

### Interpretation of subgroup profiles

The subgroup profiles provide limited evidence of systematic variation by social position.

For mental-health/learning-need status, institutional profiles were broadly similar across groups. Legal narratives were predominantly neutral/descriptive in both groups, and support-sector narratives were consistently characterised by protective/supportive roles and safety/prevention concerns. The clearest difference appeared in police narratives, where the group without a declared mental-health or learning need was more often characterised by partial/limited support, while the group with a declared need was more often characterised by protective/supportive responses. Both groups, however, remained primarily focused on safety/prevention.

Parental status produced somewhat more differentiated profiles. Among non-parents, legal narratives were strongly concentrated around consequences, while police narratives were more often characterised by institutional absence and consequence-related concerns. Among parents, police narratives were more strongly associated with protective/supportive roles and safety/prevention, while legal narratives were more oriented towards voice. Support-sector profiles remained similar across parental-status groups.

These differences are treated as exploratory signals rather than stable subgroup effects because several institution-specific subgroups contain only 5–7 annotated records.

## 22. Prepare candidate cases for targeted close reading

The structured annotation analysis identified recurrent institutional patterns as well as a small number of exploratory social-position signals.

This step prepares a candidate set for subsequent close reading. The aim is not to select quotations automatically, but to identify analytically useful records that represent:

- recurrent institution–justice–role combinations from the primary analysis;
- contrasting institutional responses within the same justice dimension;
- exploratory subgroup differences identified in the secondary analysis;
- contextual cases involving ethnicity/BME status, nationality, or physical disability where these factors may help explain institutional experience.

The resulting table is used as a case-selection aid for the next stage of integrated qualitative interpretation.

In [ ]:
# Step 22: Prepare close-reading candidate cases

# Mark recurrent primary-RQ patterns
recurrent_key = recurrent_patterns[
    ["category", "justice_dimension", "institutional_role"]
].drop_duplicates()

close_reading_candidates = ann_social.merge(
    recurrent_key.assign(primary_recurrent_pattern=True),
    on=["category", "justice_dimension", "institutional_role"],
    how="left"
)

close_reading_candidates["primary_recurrent_pattern"] = (
    close_reading_candidates["primary_recurrent_pattern"]
    .fillna(False)
)

# Mark exploratory social-position cases
close_reading_candidates["social_position_context"] = (
    (close_reading_candidates["bme_status"] == "Y")
    | (close_reading_candidates["physical_disability"] == 1)
    | (close_reading_candidates["nationality_summary"] != "British national")
)

# Mark subgroup cases worth checking from Step 20–21
close_reading_candidates["subgroup_signal_case"] = False

# Police: mental-health / learning-need contrast
close_reading_candidates.loc[
    (close_reading_candidates["category"] == "police")
    & (
        close_reading_candidates["institutional_role"]
        .isin(["partial_limited_support", "protective_supportive"])
    ),
    "subgroup_signal_case"
] = True

# Police and legal: parental-status contrast
close_reading_candidates.loc[
    (
        close_reading_candidates["category"].isin(["police", "legal"])
    )
    & (
        close_reading_candidates["parental_status"].isin(["Yes", "No"])
    ),
    "subgroup_signal_case"
] = True

# Overall candidate flag
close_reading_candidates["close_reading_candidate"] = (
    close_reading_candidates["primary_recurrent_pattern"]
    | close_reading_candidates["social_position_context"]
    | close_reading_candidates["subgroup_signal_case"]
)

candidate_cols = [
    "annotation_id",
    "transcript_id",
    "category",
    "keyword",
    "context",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction",
    "dominant_topic",
    "mental_health_learning_need",
    "parental_status",
    "bme_status",
    "nationality_summary",
    "physical_disability",
    "primary_recurrent_pattern",
    "social_position_context",
    "subgroup_signal_case",
    "close_reading_candidate"
]

candidate_table = (
    close_reading_candidates[
        close_reading_candidates["close_reading_candidate"]
    ][candidate_cols]
    .sort_values(
        [
            "category",
            "primary_recurrent_pattern",
            "subgroup_signal_case"
        ],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

print("Close-reading candidate records:", len(candidate_table))
print("Unique transcripts represented:", candidate_table["transcript_id"].nunique())

display(candidate_table.head(20))

In [ ]:
# Step 23: Build a targeted case-selection table

selection_pool = close_reading_candidates.copy()

# ---------------------------------------------------------
# 1. Primary RQ: institution × role × justice combinations
# ---------------------------------------------------------

primary_pool = (
    selection_pool[
        selection_pool["primary_recurrent_pattern"]
    ]
    .groupby(
        ["category", "justice_dimension", "institutional_role"],
        dropna=False
    )
    .agg(
        candidate_n=("annotation_id", "size"),
        transcript_n=("transcript_id", "nunique"),
        annotation_ids=("annotation_id", lambda x: list(x)),
        transcript_ids=("transcript_id", lambda x: list(x))
    )
    .reset_index()
    .sort_values(
        ["category", "candidate_n"],
        ascending=[True, False]
    )
)

print("=== PRIMARY-RQ CLOSE-READING POOL ===")
display(primary_pool)


# ---------------------------------------------------------
# 2. Contrasting evaluative cases
# ---------------------------------------------------------

evaluation_pool = (
    selection_pool
    .groupby(
        ["category", "evaluative_direction"],
        dropna=False
    )
    .agg(
        candidate_n=("annotation_id", "size"),
        transcript_n=("transcript_id", "nunique"),
        annotation_ids=("annotation_id", lambda x: list(x))
    )
    .reset_index()
    .sort_values(
        ["category", "candidate_n"],
        ascending=[True, False]
    )
)

print("\n=== EVALUATIVE CONTRAST POOL ===")
display(evaluation_pool)


# ---------------------------------------------------------
# 3. Exploratory subgroup cases
# ---------------------------------------------------------

subgroup_pool = selection_pool[
    selection_pool["social_position_context"]
    | selection_pool["subgroup_signal_case"]
][[
    "annotation_id",
    "transcript_id",
    "category",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction",
    "mental_health_learning_need",
    "parental_status",
    "bme_status",
    "nationality_summary",
    "physical_disability"
]].drop_duplicates()

print("\n=== EXPLORATORY SUBGROUP POOL ===")
print("Records:", len(subgroup_pool))
print("Transcripts:", subgroup_pool["transcript_id"].nunique())

display(subgroup_pool.head(30))

## Step 24. Construct a targeted close-reading pool

The previous step identified a broad pool of close-reading candidates covering recurrent institution × justice-dimension × institutional-role patterns, evaluative contrasts, and exploratory social-position characteristics.

This step narrows that pool to a smaller set of analytically important patterns for subsequent close reading. Selection is purposive rather than purely frequency-based. Priority is given to patterns that:

1. represent recurrent or prominent institutional narratives identified in the manual annotation;
2. capture important contrasts within the same institutional category, such as supportive versus limited, inaccessible, absent, or harmful institutional responses;
3. cover the main justice dimensions through which police, legal institutions, and support-sector organisations are described; and
4. provide cases capable of explaining or qualifying the broader patterns identified through KWIC, TF-IDF, NMF, and manual annotation.

The resulting pool is therefore not treated as a statistically representative subsample. It is an analytically targeted set of cases from which information-rich examples can be selected for close reading.

Social-position metadata are retained in the candidate table so that potentially relevant subgroup patterns can be considered during case selection, while the primary research question remains the main basis for selection.

In [ ]:
# Step 24: Build a targeted close-reading selection pool

priority_patterns = pd.DataFrame([
    # Legal
    ["legal", "consequences", "neutral_descriptive"],
    ["legal", "consequences", "partial_limited_support"],
    ["legal", "voice", "harmful_retraumatising"],
    ["legal", "safety_prevention", "protective_supportive"],

    # Police
    ["police", "safety_prevention", "protective_supportive"],
    ["police", "safety_prevention", "partial_limited_support"],
    ["police", "consequences", "absent_unavailable"],
    ["police", "dignity", "harmful_retraumatising"],

    # Support sector
    ["support_sector", "safety_prevention", "protective_supportive"],
    ["support_sector", "recognition", "protective_supportive"],
    ["support_sector", "connectedness_recovery", "protective_supportive"],
    ["support_sector", "connectedness_recovery", "partial_limited_support"],
    ["support_sector", "connectedness_recovery", "bureaucratic_inaccessible"],
], columns=[
    "category",
    "justice_dimension",
    "institutional_role"
])

targeted_pool = ann_social.merge(
    priority_patterns.assign(priority_pattern=True),
    on=[
        "category",
        "justice_dimension",
        "institutional_role"
    ],
    how="inner"
)

targeted_cols = [
    "annotation_id",
    "transcript_id",
    "category",
    "keyword",
    "context",
    "institutional_role",
    "justice_dimension",
    "evaluative_direction",
    "dominant_topic",
    "dominant_topic_weight",
    "mental_health_learning_need",
    "parental_status",
    "bme_status",
    "nationality_summary",
    "physical_disability"
]

targeted_pool = (
    targeted_pool[targeted_cols]
    .sort_values(
        [
            "category",
            "justice_dimension",
            "institutional_role"
        ]
    )
    .reset_index(drop=True)
)

print("=== TARGETED CLOSE-READING POOL ===")
print("Candidate annotations:", len(targeted_pool))
print("Unique transcripts:", targeted_pool["transcript_id"].nunique())

print("\nCandidates by priority pattern:")
display(
    targeted_pool
    .groupby(
        ["category", "justice_dimension", "institutional_role"]
    )
    .agg(
        candidate_n=("annotation_id", "size"),
        transcript_n=("transcript_id", "nunique"),
        annotation_ids=("annotation_id", lambda x: list(x))
    )
    .reset_index()
)

print("\nCandidate records:")
display(targeted_pool)

## Step 25. Export close-reading materials

The targeted close-reading pool contains 39 annotated records from 35 transcripts, covering 13 priority institution–justice–role patterns.

These records are exported as an intermediate analytical dataset for the next stage. The exported file preserves annotation labels, NMF topic information, and selected social-position metadata so that close reading can compare computational patterns, manual interpretation, and participant context without altering the original annotation dataset.

No final quotation or case selection is made at this stage. Final close-reading cases will be selected after reviewing the full context of the candidate passages.

In [ ]:
# Step 25: Export targeted close-reading materials


# Export candidate-level dataset
candidate_output = (
    OUTPUT_DIR
    / "step25_targeted_close_reading_pool.xlsx"
)

targeted_pool.to_excel(
    candidate_output,
    index=False
)


# Export priority-pattern summary
pattern_output = (
    OUTPUT_DIR
    / "step25_close_reading_pattern_summary.csv"
)

priority_pattern_summary = (
    targeted_pool
    .groupby(
        [
            "category",
            "justice_dimension",
            "institutional_role"
        ]
    )
    .agg(
        candidate_n=("annotation_id", "size"),
        transcript_n=("transcript_id", "nunique")
    )
    .reset_index()
)

priority_pattern_summary.to_csv(
    pattern_output,
    index=False
)

print("Candidate pool saved to:")
print(candidate_output)

print("\nPattern summary saved to:")
print(pattern_output)

print("\nExport summary:")
print("Candidate annotations:", len(targeted_pool))
print("Unique transcripts:", targeted_pool["transcript_id"].nunique())
print("Priority patterns:", len(priority_pattern_summary))